# 💧 DWFA — Étude d'accès à l'eau potable
## Prétraitement des données

**Mission :** Consultant data analyst pour l'ONG DWFA (Drinking Water For All)  
**Objectif :** Nettoyer, transformer et exporter les données brutes pour Power BI  
**Auteur :** [Ton Prénom Nom]  
**Date :** Juin 2026  

---

In [1]:
# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

print("✅ Librairies chargées")
print(f"   pandas : {pd.__version__}")
print(f"   numpy  : {np.__version__}")

✅ Librairies chargées
   pandas : 3.0.3
   numpy  : 2.4.6


## 2. Structure du projet

Les données transitent par 3 niveaux :

| Dossier | Rôle |
|---|---|
| `data/raw/` | Données brutes originales — jamais modifiées |
| `data/staging/` | Tables nettoyées individuellement |
| `data/mart/` | Tables finales jointes + indicateurs calculés → Power BI |

In [37]:

# CHEMINS

BASE_DIR = os.path.dirname(os.path.abspath("__file__"))

RAW     = os.path.join(BASE_DIR, "data", "raw")
STAGING = os.path.join(BASE_DIR, "data", "staging")
MART    = os.path.join(BASE_DIR, "data", "mart")

for folder in [STAGING, MART]:
    os.makedirs(folder, exist_ok=True)

print("✅ Chemins configurés")
print(f"   RAW     : {RAW}")
print(f"   STAGING : {STAGING}")
print(f"   MART    : {MART}")

✅ Chemins configurés
   RAW     : c:\Users\Candidat\Desktop\Informatique\OPEN CLASSROOM\P10 GLOBAL\notebooks\data\raw
   STAGING : c:\Users\Candidat\Desktop\Informatique\OPEN CLASSROOM\P10 GLOBAL\notebooks\data\staging
   MART    : c:\Users\Candidat\Desktop\Informatique\OPEN CLASSROOM\P10 GLOBAL\notebooks\data\mart


## 3. Chargement des données brutes

Chargement des 5 fichiers sources sans aucune modification.  
Toute transformation se fera dans les étapes suivantes.

In [3]:
# ============================================================
# CHARGEMENT
# ============================================================

basic      = pd.read_csv(os.path.join(RAW, "BasicAndSafelyManagedDrinkingWaterServices.csv"))
mortality  = pd.read_csv(os.path.join(RAW, "MortalityRateAttributedToWater.csv"))
political  = pd.read_csv(os.path.join(RAW, "PoliticalStability.csv"))
population = pd.read_csv(os.path.join(RAW, "Population.csv"))
region     = pd.read_csv(os.path.join(RAW, "RegionCountry.csv"))

datasets = {
    "basic"      : basic,
    "mortality"  : mortality,
    "political"  : political,
    "population" : population,
    "region"     : region,
}

print("✅ Fichiers chargés\n")
for name, df in datasets.items():
    print(f"  {name:<12} → {df.shape[0]:>6} lignes | {df.shape[1]} colonnes")

✅ Fichiers chargés

  basic        →  10476 lignes | 5 colonnes
  mortality    →    549 lignes | 5 colonnes
  political    →   3526 lignes | 4 colonnes
  population   →  20914 lignes | 4 colonnes
  region       →    194 lignes | 2 colonnes


## 4. Audit qualité des données brutes

Avant tout nettoyage, on audite chaque table pour identifier :
- Le nombre de lignes et colonnes
- Les doublons
- Les valeurs manquantes (count + %)
- Les types de colonnes

> 🎯 Cette fonction sera appelée **avant et après chaque nettoyage** pour tracer les transformations.

In [4]:
# ============================================================
# FONCTION AUDIT
# ============================================================

def audit(df, name="DataFrame"):
    """Rapport qualité complet sur un DataFrame."""
    print(f"\n{'='*55}")
    print(f"  AUDIT — {name}")
    print(f"{'='*55}")
    print(f"  Shape    : {df.shape[0]} lignes × {df.shape[1]} colonnes")
    print(f"  Doublons : {df.duplicated().sum()}")
    print(f"\n  Valeurs manquantes :")

    nan_df = df.isnull().sum().reset_index()
    nan_df.columns = ['Colonne', 'NaN']
    nan_df['%'] = (nan_df['NaN'] / len(df) * 100).round(1)
    nan_df = nan_df[nan_df['NaN'] > 0]

    if nan_df.empty:
        print("    → Aucune valeur manquante ✅")
    else:
        for _, row in nan_df.iterrows():
            print(f"    {row['Colonne']:<50} {row['NaN']:>6} NaN ({row['%']}%)")

    print(f"\n  Types :")
    for col, dtype in df.dtypes.items():
        print(f"    {col:<50} {str(dtype)}")

    print(f"\n  Aperçu :")
    display(df.head(3))

print("✅ Fonction audit définie")

✅ Fonction audit définie


In [5]:
# ============================================================
# AUDIT DES DONNÉES BRUTES
# ============================================================

for name, df in datasets.items():
    audit(df, name)


  AUDIT — basic
  Shape    : 10476 lignes × 5 colonnes
  Doublons : 0

  Valeurs manquantes :
    Population using at least basic drinking-water services (%)   1061 NaN (10.1%)
    Population using safely managed drinking-water services (%)   7190 NaN (68.6%)

  Types :
    Year                                               int64
    Country                                            str
    Granularity                                        str
    Population using at least basic drinking-water services (%) float64
    Population using safely managed drinking-water services (%) float64

  Aperçu :


,Year,Country,Granularity,Population using at least basic drinking-water services (%),Population using safely managed drinking-water services (%)
0,2000,Afghanistan,Rural,21.62,NaN
1,2000,Afghanistan,Total,27.77,NaN
2,2000,Afghanistan,Urban,49.49,NaN



  AUDIT — mortality
  Shape    : 549 lignes × 5 colonnes
  Doublons : 0

  Valeurs manquantes :
    WASH deaths                                           366 NaN (66.7%)

  Types :
    Year                                               int64
    Country                                            str
    Granularity                                        str
    Mortality rate attributed to exposure to unsafe WASH services float64
    WASH deaths                                        float64

  Aperçu :


,Year,Country,Granularity,Mortality rate attributed to exposure to unsafe WASH services,WASH deaths
0,2016,Afghanistan,Female,15.31,NaN
1,2016,Afghanistan,Male,12.61,NaN
2,2016,Afghanistan,Total,13.92,4824.35



  AUDIT — political
  Shape    : 3526 lignes × 4 colonnes
  Doublons : 0

  Valeurs manquantes :
    → Aucune valeur manquante ✅

  Types :
    Country                                            str
    Year                                               int64
    Political_Stability                                float64
    Granularity                                        str

  Aperçu :


,Country,Year,Political_Stability,Granularity
0,Afghanistan,2000,-2.44,Total
1,Afghanistan,2002,-2.04,Total
2,Afghanistan,2003,-2.20,Total



  AUDIT — population
  Shape    : 20914 lignes × 4 colonnes
  Doublons : 0

  Valeurs manquantes :
    → Aucune valeur manquante ✅

  Types :
    Country                                            str
    Granularity                                        str
    Year                                               int64
    Population                                         float64

  Aperçu :


,Country,Granularity,Year,Population
0,Afghanistan,Total,2000,20779.95
1,Afghanistan,Male,2000,10689.51
2,Afghanistan,Female,2000,10090.45



  AUDIT — region
  Shape    : 194 lignes × 2 colonnes
  Doublons : 0

  Valeurs manquantes :
    → Aucune valeur manquante ✅

  Types :
    REGION (DISPLAY)                                   str
    COUNTRY (DISPLAY)                                  str

  Aperçu :


,REGION (DISPLAY),COUNTRY (DISPLAY)
0,Europe,Albania
1,Europe,Andorra
2,Europe,Armenia


## 5. Nettoyage des tables (Staging)

Chaque table suit ce protocole :
1. **Audit avant** — état initial
2. **Nettoyage** — renommage, pivot, filtrage, NaN
3. **Audit après** — vérification des transformations
4. **Export staging** — sauvegarde intermédiaire

---

### 5.1 BasicAndSafelyManaged

On renomme les colonnes, on pivote la table pour avoir une ligne par pays/année
avec les granularités (Total/Urban/Rural) en colonnes, puis on exporte en staging.

In [6]:
# ============================================================
# STAGING — BasicAndSafelyManaged
# ============================================================

# --- Audit avant ---
audit(basic, "basic — AVANT")

# --- Nettoyage ---
basic_clean = basic.copy()

# Renommage des colonnes
basic_clean.columns = ['year', 'country', 'granularity', 'basic_pct', 'safely_managed_pct']

# On garde uniquement Total / Urban / Rural
basic_clean = basic_clean[basic_clean['granularity'].isin(['Total', 'Urban', 'Rural'])]

# Pivot : une ligne par pays/année, colonnes par granularité
basic_pivot = basic_clean.pivot_table(
    index=['country', 'year'],
    columns='granularity',
    values=['basic_pct', 'safely_managed_pct']
)

# Aplatir les colonnes multi-niveaux
basic_pivot.columns = [f"{val}_{gran.lower()}" for val, gran in basic_pivot.columns]
basic_pivot = basic_pivot.reset_index()

# --- Audit après ---
audit(basic_pivot, "basic — APRÈS")

# --- Export staging ---
basic_pivot.to_csv(os.path.join(STAGING, "basic.csv"), index=False, sep=';', decimal=',', encoding='utf-8-sig')
print("\n✅ Export staging/basic.csv")


  AUDIT — basic — AVANT
  Shape    : 10476 lignes × 5 colonnes
  Doublons : 0

  Valeurs manquantes :
    Population using at least basic drinking-water services (%)   1061 NaN (10.1%)
    Population using safely managed drinking-water services (%)   7190 NaN (68.6%)

  Types :
    Year                                               int64
    Country                                            str
    Granularity                                        str
    Population using at least basic drinking-water services (%) float64
    Population using safely managed drinking-water services (%) float64

  Aperçu :


,Year,Country,Granularity,Population using at least basic drinking-water services (%),Population using safely managed drinking-water services (%)
0,2000,Afghanistan,Rural,21.62,NaN
1,2000,Afghanistan,Total,27.77,NaN
2,2000,Afghanistan,Urban,49.49,NaN



  AUDIT — basic — APRÈS
  Shape    : 3455 lignes × 8 colonnes
  Doublons : 0

  Valeurs manquantes :
    basic_pct_rural                                       502 NaN (14.5%)
    basic_pct_total                                         6 NaN (0.2%)
    basic_pct_urban                                       442 NaN (12.8%)
    safely_managed_pct_rural                             2843 NaN (82.3%)
    safely_managed_pct_total                             1710 NaN (49.5%)
    safely_managed_pct_urban                             2526 NaN (73.1%)

  Types :
    country                                            str
    year                                               int64
    basic_pct_rural                                    float64
    basic_pct_total                                    float64
    basic_pct_urban                                    float64
    safely_managed_pct_rural                           float64
    safely_managed_pct_total                           float64
    safel

,country,year,basic_pct_rural,basic_pct_total,basic_pct_urban,safely_managed_pct_rural,safely_managed_pct_total,safely_managed_pct_urban
0,Afghanistan,2000,21.62,27.77,49.49,NaN,NaN,NaN
1,Afghanistan,2001,21.62,27.80,49.49,NaN,NaN,NaN
2,Afghanistan,2002,23.60,29.90,51.90,NaN,NaN,NaN



✅ Export staging/basic.csv


### 5.2 MortalityRate

On garde uniquement la granularité Total, on renomme les colonnes et on supprime
la colonne WASH deaths (66.7% de NaN). Attention : données disponibles pour 2016 uniquement.

In [7]:
# ============================================================
# STAGING — MortalityRate
# ============================================================

# --- Audit avant ---
audit(mortality, "mortality — AVANT")

# --- Nettoyage ---
mortality_clean = mortality.copy()

# On garde uniquement Total
mortality_clean = mortality_clean[mortality_clean['Granularity'] == 'Total']

# On supprime WASH deaths (66.7% NaN) — on garde uniquement le taux
mortality_clean = mortality_clean.drop(columns=['WASH deaths', 'Granularity'])

# Renommage
mortality_clean.columns = ['year', 'country', 'wash_mortality_rate']

# --- Audit après ---
audit(mortality_clean, "mortality — APRÈS")

# --- Export staging ---
mortality_clean.to_csv(os.path.join(STAGING, "mortality.csv"), index=False, sep=';', decimal=',', encoding='utf-8-sig')
print("\n✅ Export staging/mortality.csv")


  AUDIT — mortality — AVANT
  Shape    : 549 lignes × 5 colonnes
  Doublons : 0

  Valeurs manquantes :
    WASH deaths                                           366 NaN (66.7%)

  Types :
    Year                                               int64
    Country                                            str
    Granularity                                        str
    Mortality rate attributed to exposure to unsafe WASH services float64
    WASH deaths                                        float64

  Aperçu :


,Year,Country,Granularity,Mortality rate attributed to exposure to unsafe WASH services,WASH deaths
0,2016,Afghanistan,Female,15.31,NaN
1,2016,Afghanistan,Male,12.61,NaN
2,2016,Afghanistan,Total,13.92,4824.35



  AUDIT — mortality — APRÈS
  Shape    : 183 lignes × 3 colonnes
  Doublons : 0

  Valeurs manquantes :
    → Aucune valeur manquante ✅

  Types :
    year                                               int64
    country                                            str
    wash_mortality_rate                                float64

  Aperçu :


,year,country,wash_mortality_rate
2,2016,Afghanistan,13.92
5,2016,Albania,0.17
8,2016,Algeria,1.87



✅ Export staging/mortality.csv


### 5.3 PoliticalStability

On garde uniquement la granularité Total, on renomme les colonnes.
Table déjà propre — aucun NaN détecté lors de l'audit initial.

In [8]:
# ============================================================
# STAGING — PoliticalStability
# ============================================================

# --- Audit avant ---
audit(political, "political — AVANT")

# --- Nettoyage ---
political_clean = political.copy()

# On garde uniquement Total
political_clean = political_clean[political_clean['Granularity'] == 'Total']

# On supprime la colonne Granularity devenue inutile
political_clean = political_clean.drop(columns=['Granularity'])

# Renommage
political_clean.columns = ['country', 'year', 'political_stability']

# --- Audit après ---
audit(political_clean, "political — APRÈS")

# --- Export staging ---
political_clean.to_csv(os.path.join(STAGING, "political.csv"), index=False, sep=';', decimal=',', encoding='utf-8-sig')
print("\n✅ Export staging/political.csv")


  AUDIT — political — AVANT
  Shape    : 3526 lignes × 4 colonnes
  Doublons : 0

  Valeurs manquantes :
    → Aucune valeur manquante ✅

  Types :
    Country                                            str
    Year                                               int64
    Political_Stability                                float64
    Granularity                                        str

  Aperçu :


,Country,Year,Political_Stability,Granularity
0,Afghanistan,2000,-2.44,Total
1,Afghanistan,2002,-2.04,Total
2,Afghanistan,2003,-2.20,Total



  AUDIT — political — APRÈS
  Shape    : 3526 lignes × 3 colonnes
  Doublons : 0

  Valeurs manquantes :
    → Aucune valeur manquante ✅

  Types :
    country                                            str
    year                                               int64
    political_stability                                float64

  Aperçu :


,country,year,political_stability
0,Afghanistan,2000,-2.44
1,Afghanistan,2002,-2.04
2,Afghanistan,2003,-2.20



✅ Export staging/political.csv


### 5.4 Population

On pivote la table pour avoir une ligne par pays/année avec les granularités
(Total/Urban/Rural) en colonnes. On filtre les 45 territoires absents de RegionCountry
pour éviter les doublons lors des jointures.

In [9]:
# ============================================================
# STAGING — Population
# ============================================================

# --- Audit avant ---
audit(population, "population — AVANT")

# --- Nettoyage ---
population_clean = population.copy()

# Renommage
population_clean.columns = ['country', 'granularity', 'year', 'population']

# On garde uniquement Total / Urban / Rural
population_clean = population_clean[population_clean['granularity'].isin(['Total', 'Urban', 'Rural'])]

# Filtrage des 45 territoires absents de RegionCountry
pays_region = set(region['COUNTRY (DISPLAY)'].unique())
population_clean = population_clean[population_clean['country'].isin(pays_region)]

# Pivot : une ligne par pays/année
population_pivot = population_clean.pivot_table(
    index=['country', 'year'],
    columns='granularity',
    values='population'
)

# Aplatir les colonnes
population_pivot.columns = [f"pop_{gran.lower()}" for gran in population_pivot.columns]
population_pivot = population_pivot.reset_index()

# --- Audit après ---
audit(population_pivot, "population — APRÈS")

# --- Export staging ---
population_pivot.to_csv(os.path.join(STAGING, "population.csv"), index=False, sep=';', decimal=',', encoding='utf-8-sig')
print("\n✅ Export staging/population.csv")


  AUDIT — population — AVANT
  Shape    : 20914 lignes × 4 colonnes
  Doublons : 0

  Valeurs manquantes :
    → Aucune valeur manquante ✅

  Types :
    Country                                            str
    Granularity                                        str
    Year                                               int64
    Population                                         float64

  Aperçu :


,Country,Granularity,Year,Population
0,Afghanistan,Total,2000,20779.95
1,Afghanistan,Male,2000,10689.51
2,Afghanistan,Female,2000,10090.45



  AUDIT — population — APRÈS
  Shape    : 3631 lignes × 5 colonnes
  Doublons : 0

  Valeurs manquantes :
    → Aucune valeur manquante ✅

  Types :
    country                                            str
    year                                               int64
    pop_rural                                          float64
    pop_total                                          float64
    pop_urban                                          float64

  Aperçu :


,country,year,pop_rural,pop_total,pop_urban
0,Afghanistan,2000,15657.47,20779.95,4436.28
1,Afghanistan,2001,16318.32,21606.99,4648.14
2,Afghanistan,2002,17086.91,22600.77,4893.01



✅ Export staging/population.csv


### 5.5 RegionCountry

On renomme simplement les colonnes pour harmoniser avec les autres tables.
Table de référence géographique — aucune transformation nécessaire.

In [10]:
# ============================================================
# STAGING — RegionCountry
# ============================================================

# --- Audit avant ---
audit(region, "region — AVANT")

# --- Nettoyage ---
region_clean = region.copy()

# Renommage
region_clean.columns = ['region', 'country']

# --- Audit après ---
audit(region_clean, "region — APRÈS")

# --- Export staging ---
region_clean.to_csv(os.path.join(STAGING, "region.csv"), index=False, sep=';', decimal=',', encoding='utf-8-sig')
print("\n✅ Export staging/region.csv")


  AUDIT — region — AVANT
  Shape    : 194 lignes × 2 colonnes
  Doublons : 0

  Valeurs manquantes :
    → Aucune valeur manquante ✅

  Types :
    REGION (DISPLAY)                                   str
    COUNTRY (DISPLAY)                                  str

  Aperçu :


,REGION (DISPLAY),COUNTRY (DISPLAY)
0,Europe,Albania
1,Europe,Andorra
2,Europe,Armenia



  AUDIT — region — APRÈS
  Shape    : 194 lignes × 2 colonnes
  Doublons : 0

  Valeurs manquantes :
    → Aucune valeur manquante ✅

  Types :
    region                                             str
    country                                            str

  Aperçu :


,region,country
0,Europe,Albania
1,Europe,Andorra
2,Europe,Armenia



✅ Export staging/region.csv


### 5.6 Récapitulatif des NaN — tables staging

In [11]:
# ============================================================
# RÉCAP NaN — TOUTES LES TABLES STAGING
# ============================================================

tables_staging = {
    "basic"      : basic_pivot,
    "mortality"  : mortality_clean,
    "political"  : political_clean,
    "population" : population_pivot,
    "region"     : region_clean,
}

print(f"{'Table':<15} {'Colonne':<45} {'NaN':>6} {'%':>6}")
print("="*75)

for name, df in tables_staging.items():
    nan_df = df.isnull().sum().reset_index()
    nan_df.columns = ['Colonne', 'NaN']
    nan_df['%'] = (nan_df['NaN'] / len(df) * 100).round(1)
    nan_df = nan_df[nan_df['NaN'] > 0]

    if nan_df.empty:
        print(f"{name:<15} {'→ Aucun NaN ✅':<45}")
    else:
        for _, row in nan_df.iterrows():
            print(f"{name:<15} {row['Colonne']:<45} {row['NaN']:>6} {row['%']:>5}%")
    print("-"*75)

Table           Colonne                                          NaN      %
basic           basic_pct_rural                                  502  14.5%
basic           basic_pct_total                                    6   0.2%
basic           basic_pct_urban                                  442  12.8%
basic           safely_managed_pct_rural                        2843  82.3%
basic           safely_managed_pct_total                        1710  49.5%
basic           safely_managed_pct_urban                        2526  73.1%
---------------------------------------------------------------------------
mortality       → Aucun NaN ✅                                
---------------------------------------------------------------------------
political       → Aucun NaN ✅                                
---------------------------------------------------------------------------
population      → Aucun NaN ✅                                
------------------------------------------------------

### 5.7 Vérification population mondiale

In [12]:
# ============================================================
# VÉRIFICATION POPULATION MONDIALE
# ============================================================

# Population mondiale en 2018 après nettoyage
pop_2018 = population_pivot[
    (population_pivot['year'] == 2018)
]['pop_total'].sum()

print(f"Population mondiale 2018 (après nettoyage) : {pop_2018/1000:.2f} milliards")

# Comparaison avec les données brutes
pop_2018_brut = population[
    (population['Year'] == 2018) & 
    (population['Granularity'] == 'Total')
]['Population'].sum()

print(f"Population mondiale 2018 (données brutes)  : {pop_2018_brut/1000:.2f} milliards")
print(f"\nDifférence : {(pop_2018_brut - pop_2018)/1000:.2f} milliards supprimés ✅")

Population mondiale 2018 (après nettoyage) : 7616.34 milliards
Population mondiale 2018 (données brutes)  : 9090.75 milliards

Différence : 1474.41 milliards supprimés ✅


## 6. Jointures (Mart)

On assemble les tables staging en une table principale.
Clé de jointure : **country + year** combinés en une colonne concaténée
car Power BI ne supporte pas les jointures sur plusieurs colonnes simultanément.

In [13]:
# ============================================================
# MART — Jointures
# ============================================================

# --- Création de la clé de jointure country_year ---
basic_pivot['country_year']      = basic_pivot['country']      + '_' + basic_pivot['year'].astype(str)
population_pivot['country_year'] = population_pivot['country'] + '_' + population_pivot['year'].astype(str)
political_clean['country_year']  = political_clean['country']  + '_' + political_clean['year'].astype(str)

# --- Jointure principale : basic + population ---
mart = basic_pivot.merge(population_pivot, on=['country', 'year', 'country_year'], how='inner')

# --- Ajout stabilité politique ---
mart = mart.merge(political_clean, on=['country', 'year', 'country_year'], how='left')

# --- Ajout région ---
mart = mart.merge(region_clean, on='country', how='left')

# --- Ajout mortalité (2016 uniquement) ---
mortality_clean['country_year'] = mortality_clean['country'] + '_' + mortality_clean['year'].astype(str)
mart = mart.merge(mortality_clean, on=['country', 'year', 'country_year'], how='left')

# --- Audit ---
audit(mart, "mart — après jointures")


  AUDIT — mart — après jointures
  Shape    : 3418 lignes × 15 colonnes
  Doublons : 0

  Valeurs manquantes :
    basic_pct_rural                                       502 NaN (14.7%)
    basic_pct_total                                         6 NaN (0.2%)
    basic_pct_urban                                       442 NaN (12.9%)
    safely_managed_pct_rural                             2830 NaN (82.8%)
    safely_managed_pct_total                             1697 NaN (49.6%)
    safely_managed_pct_urban                             2513 NaN (73.5%)
    political_stability                                   281 NaN (8.2%)
    wash_mortality_rate                                  3236 NaN (94.7%)

  Types :
    country                                            str
    year                                               int64
    basic_pct_rural                                    float64
    basic_pct_total                                    float64
    basic_pct_urban                      

,country,year,basic_pct_rural,basic_pct_total,basic_pct_urban,safely_managed_pct_rural,safely_managed_pct_total,safely_managed_pct_urban,country_year,pop_rural,pop_total,pop_urban,political_stability,region,wash_mortality_rate
0,Afghanistan,2000,21.62,27.77,49.49,NaN,NaN,NaN,Afghanistan_2000,15657.47,20779.95,4436.28,-2.44,Eastern Mediterranean,NaN
1,Afghanistan,2001,21.62,27.80,49.49,NaN,NaN,NaN,Afghanistan_2001,16318.32,21606.99,4648.14,NaN,Eastern Mediterranean,NaN
2,Afghanistan,2002,23.60,29.90,51.90,NaN,NaN,NaN,Afghanistan_2002,17086.91,22600.77,4893.01,-2.04,Eastern Mediterranean,NaN


## 7. Indicateurs calculés

Création de 3 indicateurs dérivés pour les 3 domaines d'expertise DWFA :
- **Domaine 1** : % population urbaine
- **Domaine 2** : écart entre accès basique et safely managed
- **Domaine 3** : score d'efficacité gouvernementale (mortalité + accès eau)

In [14]:
# ============================================================
# FEATURE ENGINEERING — Domaine 1
# % population urbaine
# ============================================================

mart['urban_pct'] = (mart['pop_urban'] / mart['pop_total'] * 100).round(2)

print("✅ urban_pct créé")
print(mart['urban_pct'].describe())

✅ urban_pct créé
count   3418.00
mean      56.31
std       23.10
min        8.27
25%       36.73
50%       56.26
75%       75.00
max      108.45
Name: urban_pct, dtype: float64


### 7.1.1 Vérification anomalies urban_pct > 100%
Certains pays (ex : Érythrée) ont pop_urban + pop_rural > pop_total
en raison d'incohérences dans les modèles statistiques FAO.
Ces valeurs sont plafonnées à 100% — limite assumée et documentée.

In [15]:
print(mart[mart['urban_pct'] > 100][['country', 'year', 'pop_urban', 'pop_total', 'urban_pct']])

        country  year  pop_urban  pop_total  urban_pct
1635     Kuwait  2001    2107.42    2103.28     100.20
1636     Kuwait  2002    2143.83    2137.00     100.32
1637     Kuwait  2003    2169.12    2161.63     100.35
1638     Kuwait  2004    2207.94    2200.49     100.34
1639     Kuwait  2005    2276.62    2270.20     100.28
1640     Kuwait  2006    2377.26    2373.67     100.15
1644     Kuwait  2010    2998.08    2991.88     100.21
1645     Kuwait  2011    3191.05    3168.06     100.73
1646     Kuwait  2012    3395.56    3348.85     101.39
1647     Kuwait  2013    3598.39    3526.38     102.04
1648     Kuwait  2014    3782.45    3690.94     102.48
1649     Kuwait  2015    3935.79    3835.59     102.61
1650     Kuwait  2016    4052.58    3956.88     102.42
1651     Kuwait  2017    4136.53    4056.10     101.98
2017     Monaco  2006      34.41      34.19     100.64
2018     Monaco  2007      35.11      34.52     101.70
2019     Monaco  2008      35.85      34.87     102.81
2020     M

In [16]:
# Plafonnement à 100%
mart['urban_pct'] = mart['urban_pct'].clip(upper=100)

print("✅ urban_pct plafonné à 100%")
print(mart['urban_pct'].describe())
print(f"\nVérification max : {mart['urban_pct'].max()}")

✅ urban_pct plafonné à 100%
count   3418.00
mean      56.28
std       23.03
min        8.27
25%       36.73
50%       56.26
75%       75.00
max      100.00
Name: urban_pct, dtype: float64

Vérification max : 100.0


### 7.2 Domaine 2 — Écart basique vs safely managed

Indicateur pour identifier les pays ayant des infrastructures basiques
mais pas encore de qualité. Plus l'écart est grand, plus le besoin de
modernisation est fort.

In [17]:
# ============================================================
# FEATURE ENGINEERING — Domaine 2
# Écart basique vs safely managed
# ============================================================

mart['gap_quality'] = (mart['basic_pct_total'] - mart['safely_managed_pct_total']).round(2)

print("✅ gap_quality créé")
print(mart['gap_quality'].describe())
print(f"\nNaN : {mart['gap_quality'].isnull().sum()} ({mart['gap_quality'].isnull().sum()/len(mart)*100:.1f}%)")

✅ gap_quality créé
count   1721.00
mean      17.18
std       17.93
min        0.00
25%        1.59
50%        8.90
75%       31.25
max       65.98
Name: gap_quality, dtype: float64

NaN : 1697 (49.6%)


### 7.3 Domaine 3 — Score d'efficacité gouvernementale

Indicateur composite combinant le taux de mortalité WASH normalisé
et le taux d'accès à l'eau basique. Calculé uniquement sur 2016
(seule année disponible pour la mortalité).
Formule : score = (mortality_norm + basic_pct_total / 100) / 2
Logique additive — neutre, sans effet de compensation artificielle.

In [18]:
# ============================================================
# FEATURE ENGINEERING — Domaine 3
# Score d'efficacité gouvernementale (sur 2016 uniquement)
# ============================================================

# On travaille uniquement sur les lignes 2016
mask_2016 = mart['year'] == 2016

# Normalisation min/max de la mortalité (inversée — moins c'est mieux)
val_min = mart.loc[mask_2016, 'wash_mortality_rate'].min()
val_max = mart.loc[mask_2016, 'wash_mortality_rate'].max()

mortality_norm = 1 - (mart.loc[mask_2016, 'wash_mortality_rate'] - val_min) / (val_max - val_min)

# Score final
mart.loc[mask_2016, 'gov_score'] = (
    (mortality_norm + mart.loc[mask_2016, 'basic_pct_total'] / 100) / 2
).round(4)

print("✅ gov_score créé")
print(f"Lignes 2016 avec gov_score : {mart.loc[mask_2016, 'gov_score'].notna().sum()}")
print(mart.loc[mask_2016, 'gov_score'].describe())
print(f"\nNaN total : {mart['gov_score'].isnull().sum()} ({mart['gov_score'].isnull().sum()/len(mart)*100:.1f}%)")


✅ gov_score créé
Lignes 2016 avec gov_score : 182
count   182.00
mean      0.87
std       0.18
min       0.19
25%       0.81
50%       0.97
75%       0.99
max       1.00
Name: gov_score, dtype: float64

NaN total : 3236 (94.7%)


## 8. Export final (Mart)

Export de la table principale pour Power BI.
On ajoute la colonne country_year pour les jointures dans Power BI
qui ne supporte pas les jointures sur plusieurs colonnes simultanément.

In [19]:
# ============================================================
# EXPORT MART
# ============================================================

# Vérification finale avant export
audit(mart, "mart — FINAL")

# Export
mart.to_csv(
    os.path.join(MART, "mart.csv"),
    index=False,
    sep=';',
    decimal=',',
    encoding='utf-8-sig'
)

print(f"\n✅ Export mart/mart.csv")
print(f"   {mart.shape[0]} lignes × {mart.shape[1]} colonnes")


  AUDIT — mart — FINAL
  Shape    : 3418 lignes × 18 colonnes
  Doublons : 0

  Valeurs manquantes :
    basic_pct_rural                                       502 NaN (14.7%)
    basic_pct_total                                         6 NaN (0.2%)
    basic_pct_urban                                       442 NaN (12.9%)
    safely_managed_pct_rural                             2830 NaN (82.8%)
    safely_managed_pct_total                             1697 NaN (49.6%)
    safely_managed_pct_urban                             2513 NaN (73.5%)
    political_stability                                   281 NaN (8.2%)
    wash_mortality_rate                                  3236 NaN (94.7%)
    gap_quality                                          1697 NaN (49.6%)
    gov_score                                            3236 NaN (94.7%)

  Types :
    country                                            str
    year                                               int64
    basic_pct_rural          

,country,year,basic_pct_rural,basic_pct_total,basic_pct_urban,safely_managed_pct_rural,safely_managed_pct_total,safely_managed_pct_urban,country_year,pop_rural,pop_total,pop_urban,political_stability,region,wash_mortality_rate,urban_pct,gap_quality,gov_score
0,Afghanistan,2000,21.62,27.77,49.49,NaN,NaN,NaN,Afghanistan_2000,15657.47,20779.95,4436.28,-2.44,Eastern Mediterranean,NaN,21.35,NaN,NaN
1,Afghanistan,2001,21.62,27.80,49.49,NaN,NaN,NaN,Afghanistan_2001,16318.32,21606.99,4648.14,NaN,Eastern Mediterranean,NaN,21.51,NaN,NaN
2,Afghanistan,2002,23.60,29.90,51.90,NaN,NaN,NaN,Afghanistan_2002,17086.91,22600.77,4893.01,-2.04,Eastern Mediterranean,NaN,21.65,NaN,NaN



✅ Export mart/mart.csv
   3418 lignes × 18 colonnes


## 8.1 Clé de jointure country_year — tables staging

Power BI ne supporte pas les jointures sur plusieurs colonnes simultanément.
On ajoute une colonne **country_year** (ex: "France_2015") dans chaque table
staging pour permettre les relations dans le modèle de données Power BI.

In [20]:
# ============================================================
# AJOUT country_year DANS TOUS LES STAGING
# ============================================================

# Rechargement et ajout de la clé
basic_pivot['country_year']      = basic_pivot['country']      + '_' + basic_pivot['year'].astype(str)
population_pivot['country_year'] = population_pivot['country'] + '_' + population_pivot['year'].astype(str)
political_clean['country_year']  = political_clean['country']  + '_' + political_clean['year'].astype(str)
mortality_clean['country_year']  = mortality_clean['country']  + '_' + mortality_clean['year'].astype(str)

# Réexport staging avec la clé
basic_pivot.to_csv(os.path.join(STAGING, "basic.csv"), index=False, sep=';', decimal=',', encoding='utf-8-sig')
population_pivot.to_csv(os.path.join(STAGING, "population.csv"), index=False, sep=';', decimal=',', encoding='utf-8-sig')
political_clean.to_csv(os.path.join(STAGING, "political.csv"), index=False, sep=';', decimal=',', encoding='utf-8-sig')
mortality_clean.to_csv(os.path.join(STAGING, "mortality.csv"), index=False, sep=';', decimal=',', encoding='utf-8-sig')

# Vérification
for name, df in [("basic", basic_pivot), ("population", population_pivot), 
                  ("political", political_clean), ("mortality", mortality_clean)]:
    print(f"{name:<12} → country_year présent : {'country_year' in df.columns} ✅")

basic        → country_year présent : True ✅
population   → country_year présent : True ✅
political    → country_year présent : True ✅
mortality    → country_year présent : True ✅


## 9. Export tables Power BI

On exporte les tables séparées pour Power BI en suivant
un modèle en étoile : une table centrale reliée aux dimensions
via la clé country_year.

### 9.1 Ajout colonnes no_acces — table water

On calcule le % de population sans accès à l'eau
pour Total, Urban et Rural (100 - basic_pct).

In [41]:
# ============================================================
# AJOUT colonnes no_acces — table water
# ============================================================

water['no_acces_pct']       = (100 - water['basic_pct_total']).round(2)
water['no_acces_rural_pct'] = (100 - water['basic_pct_rural']).round(2)
water['no_acces_urban_pct'] = (100 - water['basic_pct_urban']).round(2)

print("✅ Colonnes no_acces ajoutées")
print(f"   {water.shape[0]} lignes × {water.shape[1]} colonnes")
print(water[['country', 'year', 'no_acces_pct', 'no_acces_rural_pct', 'no_acces_urban_pct']].head(3))

# Réexport water avec les colonnes no_acces
water.to_csv(os.path.join(MART, "water.csv"), index=False, sep=';', decimal=',', encoding='utf-8-sig')
print(f"✅ water.csv réexporté avec no_acces — {water.shape[0]} lignes × {water.shape[1]} colonnes")

✅ Colonnes no_acces ajoutées
   3455 lignes × 12 colonnes
       country  year  no_acces_pct  no_acces_rural_pct  no_acces_urban_pct
0  Afghanistan  2000         72.23               78.38               50.51
1  Afghanistan  2001         72.20               78.38               50.51
2  Afghanistan  2002         70.10               76.40               48.10
✅ water.csv réexporté avec no_acces — 3455 lignes × 12 colonnes


### 9.1.1 Table water

Table principale accès à l'eau — basic et safely managed
par granularité (Total/Urban/Rural).

In [42]:
# ============================================================
# EXPORT — Table water
# ============================================================

water = basic_pivot.copy()

water.to_csv(os.path.join(MART, "water.csv"), index=False, sep=';', decimal=',', encoding='utf-8-sig')
print("✅ Export mart/water.csv")
print(f"   {water.shape[0]} lignes × {water.shape[1]} colonnes")

✅ Export mart/water.csv
   3455 lignes × 9 colonnes


### 9.2 Table pop_region

Table population avec % urbain et rural calculés,
enrichie avec le continent OMS via jointure avec region.

In [ ]:
# ============================================================
# TABLE — pop_region
# ============================================================

pop_region = population_pivot.merge(region_clean, on='country', how='left')

# Calcul % urbain et rural
pop_region['urban_pct'] = (pop_region['pop_urban'] / pop_region['pop_total'] * 100).round(2)
pop_region['rural_pct'] = (pop_region['pop_rural'] / pop_region['pop_total'] * 100).round(2)

# Plafonnement à 100%
pop_region['urban_pct'] = pop_region['urban_pct'].clip(upper=100)
pop_region['rural_pct'] = pop_region['rural_pct'].clip(upper=100)

print("✅ pop_region créée")
print(f"   {pop_region.shape[0]} lignes × {pop_region.shape[1]} colonnes")


✅ pop_region créée
   3631 lignes × 9 colonnes


### 9.3 Table mortality

Table mortalité WASH — disponible uniquement pour 2016.
Contient le taux de mortalité attribué à l'eau insalubre par pays.

In [23]:
# ============================================================
# TABLE — mortality
# ============================================================

mortality_export = mortality_clean.copy()

print("✅ mortality prête")
print(f"   {mortality_export.shape[0]} lignes × {mortality_export.shape[1]} colonnes")

✅ mortality prête
   183 lignes × 4 colonnes


### 9.4 Table political

Table stabilité politique par pays et par année.
Indice entre -3.3 et +1.97 — plus la valeur est haute,
plus le pays est stable.

In [24]:
# ============================================================
# TABLE — political
# ============================================================

political_export = political_clean.copy()

print("✅ political prête")
print(f"   {political_export.shape[0]} lignes × {political_export.shape[1]} colonnes")

✅ political prête
   3526 lignes × 4 colonnes


### 9.5 Table region

Table de correspondance pays → continent OMS.
6 régions : Europe, Africa, Americas, Western Pacific,
Eastern Mediterranean, South-East Asia.

In [25]:
# ============================================================
# TABLE — region
# ============================================================

region_export = region_clean.copy()

print("✅ region prête")
print(f"   {region_export.shape[0]} lignes × {region_export.shape[1]} colonnes")

✅ region prête
   194 lignes × 2 colonnes


### 9.6 Table gov_effectiveness

Table du score d'efficacité gouvernementale — calculé sur 2016 uniquement.
Combine le taux de mortalité WASH normalisé et le taux d'accès à l'eau basique.
Formule : score = (mortality_norm + basic_pct_total / 100) / 2

In [26]:
# ============================================================
# TABLE — gov_effectiveness
# ============================================================

# On part du mart, on filtre sur 2016 et on garde les colonnes utiles
gov_effectiveness = mart[mart['year'] == 2016][
    ['country', 'year', 'country_year', 'basic_pct_total', 
     'wash_mortality_rate', 'gov_score']
].dropna(subset=['gov_score']).copy()

print("✅ gov_effectiveness prête")
print(f"   {gov_effectiveness.shape[0]} lignes × {gov_effectiveness.shape[1]} colonnes")

✅ gov_effectiveness prête
   182 lignes × 6 colonnes


## 10. Conclusion

Le prétraitement est terminé. Les tables exportées dans `mart/` sont :

| Table | Lignes | Contenu |
|---|---|---|
| `water.csv` | 3 455 | Accès eau basic + safely managed |
| `pop_region.csv` | 3 631 | Population + % urbain/rural + région |
| `mortality.csv` | 183 | Mortalité WASH 2016 |
| `political.csv` | 3 526 | Stabilité politique 2000–2018 |
| `region.csv` | 194 | Mapping pays → continent OMS |
| `gov_effectiveness.csv` | 182 | Score efficacité gouvernementale 2016 |

**Limites documentées :**
- Mortalité WASH disponible uniquement pour 2016
- Safely managed : 49–82% de NaN selon granularité
- urban_pct pla

## 10. Oublis / Corrections

### 10.1 Correction table mortality
On réintègre les colonnes supprimées à tort :
wash_deaths_total, wash_mortality_rate_female,
wash_mortality_rate_male, wash_mortality_rate_total.

### 10.2 Correction table pop_region
On ajoute les colonnes female et male.

In [29]:
# ============================================================
# CORRECTION — Table mortality
# ============================================================

mortality_export = mortality.copy()

# On garde uniquement Total
mortality_export = mortality_export[mortality_export['Granularity'] == 'Total']

# On supprime uniquement Granularity
mortality_export = mortality_export.drop(columns=['Granularity'])

# Renommage
mortality_export.columns = ['year', 'country', 'wash_mortality_rate_total', 'wash_deaths_total']

# Ajout country_year
mortality_export['country_year'] = mortality_export['country'] + '_' + mortality_export['year'].astype(str)

# Audit
audit(mortality_export, "mortality — CORRIGÉE")



  AUDIT — mortality — CORRIGÉE
  Shape    : 183 lignes × 5 colonnes
  Doublons : 0

  Valeurs manquantes :
    → Aucune valeur manquante ✅

  Types :
    year                                               int64
    country                                            str
    wash_mortality_rate_total                          float64
    wash_deaths_total                                  float64
    country_year                                       str

  Aperçu :


,year,country,wash_mortality_rate_total,wash_deaths_total,country_year
2,2016,Afghanistan,13.92,4824.35,Afghanistan_2016
5,2016,Albania,0.17,4.87,Albania_2016
8,2016,Algeria,1.87,758.21,Algeria_2016


In [30]:
# ============================================================
# CORRECTION — Table pop_region
# ============================================================

pop_region = population_pivot.merge(region_clean, on='country', how='left')

# Ajout female et male depuis population brute
pop_female = population[population['Granularity'] == 'Female'][['Country', 'Year', 'Population']].rename(
    columns={'Country': 'country', 'Year': 'year', 'Population': 'female'})

pop_male = population[population['Granularity'] == 'Male'][['Country', 'Year', 'Population']].rename(
    columns={'Country': 'country', 'Year': 'year', 'Population': 'male'})

# Filtrage des 45 pays hors region
pop_female = pop_female[pop_female['country'].isin(pays_region)]
pop_male   = pop_male[pop_male['country'].isin(pays_region)]

# Jointure
pop_region = pop_region.merge(pop_female, on=['country', 'year'], how='left')
pop_region = pop_region.merge(pop_male,   on=['country', 'year'], how='left')

# Recalcul urban_pct et rural_pct
pop_region['urban_pct'] = (pop_region['pop_urban'] / pop_region['pop_total'] * 100).round(2).clip(upper=100)
pop_region['rural_pct'] = (pop_region['pop_rural'] / pop_region['pop_total'] * 100).round(2).clip(upper=100)

# Audit
audit(pop_region, "pop_region — CORRIGÉE")


  AUDIT — pop_region — CORRIGÉE
  Shape    : 3631 lignes × 11 colonnes
  Doublons : 0

  Valeurs manquantes :
    female                                                209 NaN (5.8%)
    male                                                  209 NaN (5.8%)

  Types :
    country                                            str
    year                                               int64
    pop_rural                                          float64
    pop_total                                          float64
    pop_urban                                          float64
    country_year                                       str
    region                                             str
    female                                             float64
    male                                               float64
    urban_pct                                          float64
    rural_pct                                          float64

  Aperçu :


,country,year,pop_rural,pop_total,pop_urban,country_year,region,female,male,urban_pct,rural_pct
0,Afghanistan,2000,15657.47,20779.95,4436.28,Afghanistan_2000,Eastern Mediterranean,10090.45,10689.51,21.35,75.35
1,Afghanistan,2001,16318.32,21606.99,4648.14,Afghanistan_2001,Eastern Mediterranean,10489.24,11117.75,21.51,75.52
2,Afghanistan,2002,17086.91,22600.77,4893.01,Afghanistan_2002,Eastern Mediterranean,10958.67,11642.11,21.65,75.60


In [31]:
# Quels pays/années ont des NaN sur female ?
nan_female = pop_region[pop_region['female'].isnull()][['country', 'year']].drop_duplicates()
print(f"{nan_female['country'].nunique()} pays concernés :")
print(nan_female['country'].unique())

11 pays concernés :
<StringArray>
[              'Andorra',          'Cook Islands',              'Dominica',
      'Marshall Islands',                'Monaco',                 'Nauru',
                  'Niue',                 'Palau', 'Saint Kitts and Nevis',
            'San Marino',                'Tuvalu']
Length: 11, dtype: str


In [32]:
# ============================================================
# CORRECTION — Ajout female/male dans mortality
# ============================================================

# Récupération des lignes Female et Male
mortality_female = mortality[mortality['Granularity'] == 'Female'][['Year', 'Country', 'Mortality rate attributed to exposure to unsafe WASH services']].rename(
    columns={'Year': 'year', 'Country': 'country', 
             'Mortality rate attributed to exposure to unsafe WASH services': 'wash_mortality_rate_female'})

mortality_male = mortality[mortality['Granularity'] == 'Male'][['Year', 'Country', 'Mortality rate attributed to exposure to unsafe WASH services']].rename(
    columns={'Year': 'year', 'Country': 'country',
             'Mortality rate attributed to exposure to unsafe WASH services': 'wash_mortality_rate_male'})

# Jointure avec mortality_export
mortality_export = mortality_export.merge(mortality_female, on=['country', 'year'], how='left')
mortality_export = mortality_export.merge(mortality_male,   on=['country', 'year'], how='left')

# Audit
audit(mortality_export, "mortality — FINALE")


  AUDIT — mortality — FINALE
  Shape    : 183 lignes × 7 colonnes
  Doublons : 0

  Valeurs manquantes :
    → Aucune valeur manquante ✅

  Types :
    year                                               int64
    country                                            str
    wash_mortality_rate_total                          float64
    wash_deaths_total                                  float64
    country_year                                       str
    wash_mortality_rate_female                         float64
    wash_mortality_rate_male                           float64

  Aperçu :


,year,country,wash_mortality_rate_total,wash_deaths_total,country_year,wash_mortality_rate_female,wash_mortality_rate_male
0,2016,Afghanistan,13.92,4824.35,Afghanistan_2016,15.31,12.61
1,2016,Albania,0.17,4.87,Albania_2016,0.13,0.21
2,2016,Algeria,1.87,758.21,Algeria_2016,2.20,1.73


In [33]:
# ============================================================
# CORRECTION — Renommage colonnes
# ============================================================

# region → continent dans pop_region
pop_region = pop_region.rename(columns={'region': 'continent'})

# region → continent dans region_export
region_export = region_export.rename(columns={'region': 'continent'})

# wash_mortality_rate → wash_mortality_rate_total dans gov_effectiveness
gov_effectiveness = gov_effectiveness.rename(columns={'wash_mortality_rate': 'wash_mortality_rate_total'})

# Vérification
print("pop_region :", pop_region.columns.tolist())
print("region     :", region_export.columns.tolist())
print("gov        :", gov_effectiveness.columns.tolist())

pop_region : ['country', 'year', 'pop_rural', 'pop_total', 'pop_urban', 'country_year', 'continent', 'female', 'male', 'urban_pct', 'rural_pct']
region     : ['continent', 'country']
gov        : ['country', 'year', 'country_year', 'basic_pct_total', 'wash_mortality_rate_total', 'gov_score']


In [34]:
BASE = r'c:\Users\Candidat\Desktop\Informatique\OPEN CLASSROOM\P10 GLOBAL\data\mart'

water = pd.read_csv(f'{BASE}\\water.csv', encoding='utf-8-sig', sep=';', decimal=',')
pop = pd.read_csv(f'{BASE}\\pop_region.csv', encoding='utf-8-sig', sep=';', decimal=',')
mort = pd.read_csv(f'{BASE}\\mortality.csv', encoding='utf-8-sig', sep=';', decimal=',')
pol = pd.read_csv(f'{BASE}\\political.csv', encoding='utf-8-sig', sep=';', decimal=',')
region = pd.read_csv(f'{BASE}\\region.csv', encoding='utf-8-sig', sep=';', decimal=',')

print("=== Pays avec parenthèses ===")
for name, df in [('Water', water), ('Political', pol), ('Population', pop), ('Mortalité', mort), ('Région', region)]:
    col = 'country' if 'country' in df.columns else 'Country'
    avec = [c for c in df[col].unique() if '(' in str(c)]
    print(f"{name} ({len(avec)}) : {sorted(avec)}")

print("\n=== Congo ===")
for name, df in [('Water', water), ('Political', pol), ('Population', pop), ('Mortalité', mort), ('Région', region)]:
    col = 'country' if 'country' in df.columns else 'Country'
    congo = [c for c in df[col].unique() if 'Congo' in str(c)]
    print(f"{name} : {congo}")

=== Pays avec parenthèses ===
Water (4) : ['Bolivia (Plurinational State of)', 'Iran (Islamic Republic of)', 'Micronesia (Federated States of)', 'Venezuela (Bolivarian Republic of)']
Political (4) : ['Bolivia (Plurinational State of)', 'Iran (Islamic Republic of)', 'Micronesia (Federated States of)', 'Venezuela (Bolivarian Republic of)']
Population (4) : ['Bolivia (Plurinational State of)', 'Iran (Islamic Republic of)', 'Micronesia (Federated States of)', 'Venezuela (Bolivarian Republic of)']
Mortalité (4) : ['Bolivia (Plurinational State of)', 'Iran (Islamic Republic of)', 'Micronesia (Federated States of)', 'Venezuela (Bolivarian Republic of)']
Région (4) : ['Bolivia (Plurinational State of)', 'Iran (Islamic Republic of)', 'Micronesia (Federated States of)', 'Venezuela (Bolivarian Republic of)']

=== Congo ===
Water : ['Congo', 'Democratic Republic of the Congo']
Political : ['Congo', 'Democratic Republic of the Congo']
Population : ['Congo', 'Democratic Republic of the Congo']
Morta

In [36]:
# Corrections finales sur tous les fichiers mart

corrections = {
    'Bolivia (Plurinational State of)': 'Bolivia',
    'Iran (Islamic Republic of)': 'Iran',
    'Micronesia (Federated States of)': 'Micronesia',
    'Venezuela (Bolivarian Republic of)': 'Venezuela',
    'Congo': 'Republic of Congo'
}

for name, df in [('Water', water), ('Political', pol), ('Population', pop), ('Mortalité', mort), ('Région', region)]:
    col = 'country' if 'country' in df.columns else 'Country'
    df[col] = df[col].replace(corrections)

# Vérification
print("=== Vérification après corrections ===")
for name, df in [('Water', water), ('Political', pol), ('Population', pop), ('Mortalité', mort), ('Région', region)]:
    col = 'country' if 'country' in df.columns else 'Country'
    avec = [c for c in df[col].unique() if '(' in str(c)]
    congo = [c for c in df[col].unique() if 'Congo' in str(c)]
    print(f"{name} — Parenthèses : {avec} — Congo : {congo}")

# Réexporter
BASE = r'c:\Users\Candidat\Desktop\Informatique\OPEN CLASSROOM\P10 GLOBAL\data\mart'
water.to_csv(f'{BASE}\\water.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
pop.to_csv(f'{BASE}\\pop_region.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
mort.to_csv(f'{BASE}\\mortality.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
pol.to_csv(f'{BASE}\\political.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
region.to_csv(f'{BASE}\\region.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')

print("\n✅ Tous les fichiers réexportés !")

=== Vérification après corrections ===
Water — Parenthèses : [] — Congo : ['Republic of Congo', 'Democratic Republic of the Congo']
Political — Parenthèses : [] — Congo : ['Republic of Congo', 'Democratic Republic of the Congo']
Population — Parenthèses : [] — Congo : ['Republic of Congo', 'Democratic Republic of the Congo']
Mortalité — Parenthèses : [] — Congo : ['Republic of Congo', 'Democratic Republic of the Congo']
Région — Parenthèses : [] — Congo : ['Republic of Congo', 'Democratic Republic of the Congo']

✅ Tous les fichiers réexportés !


In [35]:
# ============================================================
# EXPORT GROUPÉ FINAL — toutes les tables Power BI
# ============================================================

tables_export = {
    "water"            : water,
    "pop_region"       : pop_region,
    "mortality"        : mortality_export,
    "political"        : political_export,
    "region"           : region_export,
    "gov_effectiveness": gov_effectiveness,
}

for name, df in tables_export.items():
    path = os.path.join(MART, f"{name}.csv")
    df.to_csv(path, index=False, sep=';', decimal=',', encoding='utf-8-sig')
    print(f"✅ {name:<20} → {df.shape[0]:>5} lignes × {df.shape[1]} colonnes")

print("\n🎉 Tous les fichiers exportés dans mart/")

✅ water                →  3455 lignes × 9 colonnes
✅ pop_region           →  3631 lignes × 11 colonnes
✅ mortality            →   183 lignes × 7 colonnes
✅ political            →  3526 lignes × 4 colonnes
✅ region               →   194 lignes × 2 colonnes
✅ gov_effectiveness    →   182 lignes × 6 colonnes

🎉 Tous les fichiers exportés dans mart/
